# Evaluación del modelo y variantes

En el capítulo anterior elegimos Gradient Boosting, pero lo medimos con una sola separación de
zonas. Aquí hacemos dos cosas: repetimos la separación de zonas para saber cuánto depende el
resultado del azar, y probamos tres cambios que no necesitan datos nuevos: el barrio como
variable, modelos separados para casas y apartamentos, y otra función de pérdida. Usamos las
mismas cinco métricas del capítulo 7, con el MAPE como criterio.

Las variantes se comparan solo dentro del conjunto de entrenamiento. La prueba final se usa una
sola vez, al terminar.

In [1]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "4")   # evita que XGBoost y scikit-learn se bloqueen en Jupyter
from eda_utils import *
import time
import xgboost as xgb
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupKFold, ParameterSampler
from sklearn.neighbors import BallTree
from sklearn.pipeline import make_pipeline
from xgboost import XGBRegressor
from sklearn.model_selection import KFold

In [2]:
d = viv[viv.lat.notna() & viv.lon.notna() & viv.area.notna()].copy().reset_index(drop=True)
a_2022 = ipc_anio.loc[2022] / ipc_anio
d["razon_avaluo"] = d.precio * d.anio.map(a_2022) / d.avaluo
d["al_avaluo"] = d.razon_avaluo.between(0.9, 1.1).fillna(False).astype(bool)   # sin avalúo: no se puede saber
d["tipo_txt"] = np.where((d.condicion_predio == "NPH").fillna(False), "casa", "apartamento u otro")

d = d[~d.al_avaluo].reset_index(drop=True)
d["log_area"] = np.log10(d.area)
d["log_pm2"] = np.log10(d.precio_m2_real)
d["estrato_n"] = d.estrato.astype(str).map({e: i + 1 for i, e in enumerate(ESTRATOS)})
for t in ["PH_Unidad_Predial", "NPH", "PH_Matriz"]:
    d[f"tipo_{t}"] = (d.condicion_predio == t).astype(float)
d["bloque"] = (np.floor(d.lat / 0.01).astype(int).astype(str) + "_" +
               np.floor(d.lon / 0.01).astype(int).astype(str))

for f in ["igac", "observatorio", "georreferenciadas"]:
    d[f"fuente_{f}"] = (d.fuente == f).astype(float)

# La fuente del dato se usa solo para entrenar: cada fuente registra el precio a un nivel distinto
# (el IGAC, más bajo) y así el modelo lo corrige. Al valorar una vivienda nueva la fuente se fija
# en la de referencia, el observatorio, así que nadie tiene que darla.
FUENTES = ["fuente_igac", "fuente_observatorio", "fuente_georreferenciadas"]
REFERENCIA = "observatorio"
BASE = ["log_area", "habitaciones", "banios", "piso", "plantas", "edad", "estrato_n", "lat", "lon",
        "anio", "tipo_PH_Unidad_Predial", "tipo_NPH", "tipo_PH_Matriz"] + FUENTES

bloques = d.groupby("bloque").agg(ventas=("bloque", "size"),
                                  alameda=("barrio", lambda s: (s == "SECTOR ALAMEDA DEL RIO").mean()))
candidatos = bloques[bloques.alameda < 0.5].sample(frac=1, random_state=42)
bloques_prueba = candidatos.index[candidatos.ventas.cumsum() <= 0.2 * len(d)]
es_prueba = d.bloque.isin(bloques_prueba)
ent, prueba = d[~es_prueba].reset_index(drop=True), d[es_prueba].reset_index(drop=True)
y_ent, y_prueba = ent.log_precio.to_numpy(), prueba.log_precio.to_numpy()
print(f"Entrenamiento: {len(ent):,} ventas en {ent.bloque.nunique()} bloques")
print(f"Prueba final : {len(prueba):,} ventas en {prueba.bloque.nunique()} bloques")

# Variables del entorno, iguales a las del capítulo 7, donde el MAPE decidió usarlas.
# La regresión lineal se queda con las variables de la vivienda, como en el capítulo 7.
R_TIERRA_KM = 6371
osm = pd.read_csv("datos/entorno_osm.csv")
osm = osm[(osm.categoria != "rio") | (osm.nombre == "Río Magdalena")]
DISTANCIAS = {"rio": "dist_rio", "via_principal": "dist_via", "transmetro": "dist_transmetro",
              "centro_comercial": "dist_centro_comercial", "universidad": "dist_universidad",
              "salud": "dist_salud"}
CONTEOS = {"colegio": "colegios_1km", "parque": "parques_1km", "supermercado": "supermercados_1km",
           "salud": "salud_1km"}
ventas_manzana = d.manzana.value_counts()

def entorno(datos):
    xy = np.radians(datos[["lat", "lon"]].to_numpy())
    salida = pd.DataFrame(index=datos.index)
    for cat, col in DISTANCIAS.items():
        arbol = BallTree(np.radians(osm.loc[osm.categoria == cat, ["lat", "lon"]].to_numpy()), metric="haversine")
        salida[col] = arbol.query(xy, k=1)[0][:, 0] * R_TIERRA_KM
    for cat, col in CONTEOS.items():
        arbol = BallTree(np.radians(osm.loc[osm.categoria == cat, ["lat", "lon"]].to_numpy()), metric="haversine")
        salida[col] = arbol.query_radius(xy, r=1 / R_TIERRA_KM, count_only=True)
    salida["ventas_manzana"] = np.log10(datos.manzana.map(ventas_manzana))
    return salida

OSM = list(DISTANCIAS.values()) + list(CONTEOS.values()) + ["ventas_manzana"]
ent, prueba = ent.join(entorno(ent)), prueba.join(entorno(prueba))
VIVIENDA = BASE
BASE = BASE + OSM


Entrenamiento: 24,286 ventas en 71 bloques
Prueba final : 5,645 ventas en 24 bloques


## Las métricas



In [3]:
def metricas_completas(y, p, lat=None, lon=None):
    """Las cinco métricas del capítulo 7. y y p son el log10 del precio en pesos de 2025."""
    real, pred = 10.0 ** y / 1e6, 10.0 ** p / 1e6
    return {"R²": 1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum(),
            "MAE (M)": np.mean(np.abs(real - pred)),
            "RMSE (M)": np.sqrt(np.mean((real - pred) ** 2)),
            "error abs. mediano (M)": np.median(np.abs(real - pred)),
            "MAPE (%)": np.mean(np.abs(pred / real - 1)) * 100}

## Variantes y diseño de la comparación

Partimos de Gradient Boosting con las variables y los hiperparámetros del capítulo anterior y
probamos:

- **Barrio:** precio por m² del barrio, suavizado. En entrenamiento se calcula con ventas de otros
  bloques, para que el modelo lo vea como lo verá en una zona nueva. Si el barrio no aparece en
  entrenamiento queda vacío.
- **Separado:** un modelo para casas y otro para lo demás.
- **Pérdida MAE:** cambia el error cuadrático por el error absoluto, que se ajusta a la mediana y
  le pesa menos a los precios atípicos.

Como referencia incluimos la regresión lineal y una regla ingenua: el precio mediano por m² del
estrato multiplicado por el área.

Cada variante se evalúa en 20 divisiones por zonas del conjunto de entrenamiento. Todas usan las
mismas 20 divisiones, de modo que las diferencias se pueden medir división por división. Cada una
deja aparte cerca del 20% de las ventas. El bloque de Alameda del Río se queda siempre en
entrenamiento, igual que en la prueba final.

In [4]:
# configuración elegida en el capítulo 7 con esta base
MEJORES = {'min_samples_leaf': 100, 'max_leaf_nodes': 31, 'max_iter': 300, 'max_features': 0.5, 'max_depth': None, 'learning_rate': 0.05, 'l2_regularization': 1.0}
MAE = dict(loss="absolute_error")
CON_BARRIO = BASE + ["enc_barrio"]

def ajustar(tr, te, columnas, **extra):
    modelo = HistGradientBoostingRegressor(random_state=42, **{**MEJORES, **extra})
    return modelo.fit(tr[columnas], tr.log_precio).predict(te[columnas]).astype(float)

def separar(tr, te, columnas, **extra):
    p = np.empty(len(te))
    for es_casa in [True, False]:
        mascara = (te.tipo_NPH == 1).to_numpy() == es_casa
        if mascara.any():
            p[mascara] = ajustar(tr[(tr.tipo_NPH == 1) == es_casa], te[mascara], columnas, **extra)
    return p

def con_barrio(tr, te, m=20):
    tr, te = tr.copy(), te.copy()
    media = tr.log_pm2.mean()
    def mapa(a):
        t = a.groupby("barrio").log_pm2.agg(["mean", "size"])
        return (t["mean"] * t["size"] + media * m) / (t["size"] + m)
    enc = np.full(len(tr), np.nan)
    for a, b in GroupKFold(5).split(tr, groups=tr.bloque):
        enc[b] = tr.iloc[b].barrio.map(mapa(tr.iloc[a])).astype(float).to_numpy()
    tr["enc_barrio"], te["enc_barrio"] = enc, te.barrio.map(mapa(tr)).astype(float).to_numpy()
    return tr, te

def ingenua(tr, te):
    por_estrato = tr.groupby("estrato_n").log_pm2.median()
    return te.estrato_n.map(por_estrato).fillna(tr.log_pm2.median()).to_numpy() + te.log_area.to_numpy()

def lineal(tr, te):
    f = make_pipeline(SimpleImputer(strategy="median", add_indicator=True), LinearRegression())
    return f.fit(tr[VIVIENDA], tr.log_precio).predict(te[VIVIENDA])

VARIANTES = {
    "Regla ingenua": ingenua,
    "Regresión lineal": lineal,
    "Gradient Boosting actual": lambda a, b: ajustar(a, b, BASE),
    "+ barrio": lambda a, b: ajustar(*con_barrio(a, b), CON_BARRIO),
    "Separado": lambda a, b: separar(a, b, BASE),
    "Pérdida MAE": lambda a, b: ajustar(a, b, BASE, **MAE),
    "+ barrio, separado": lambda a, b: separar(*con_barrio(a, b), CON_BARRIO),
    "+ barrio, MAE": lambda a, b: ajustar(*con_barrio(a, b), CON_BARRIO, **MAE),
}

def divisiones(datos, n=20, frac=0.2):
    bl = datos.groupby("bloque").agg(ventas=("bloque", "size"),
                                     alameda=("barrio", lambda s: (s == "SECTOR ALAMEDA DEL RIO").mean()))
    candidatos = bl[bl.alameda < 0.5]
    for s in range(n):
        c = candidatos.sample(frac=1, random_state=s)
        yield s, datos.bloque.isin(c.index[c.ventas.cumsum() <= frac * len(datos)]).to_numpy()

filas = []
for s, es_val in divisiones(ent):
    tr, te = ent[~es_val].reset_index(drop=True), ent[es_val].reset_index(drop=True)
    y = te.log_precio.to_numpy()
    casa = (te.tipo_NPH == 1).to_numpy()
    sin_barrio = te.barrio.map(tr.barrio.value_counts()).isna().mean() * 100
    for nombre, f in VARIANTES.items():
        p = f(tr, te)
        filas.append({"division": s, "modelo": nombre, "grupo": "todas", "ventas": len(te), "sin_barrio": sin_barrio,
                      **metricas_completas(y, p, te.lat.to_numpy(), te.lon.to_numpy())})
        for grupo, mascara in [("casas", casa), ("apartamentos u otros", ~casa)]:
            if mascara.sum() > 30:
                filas.append({"division": s, "modelo": nombre, "grupo": grupo, "ventas": int(mascara.sum()),
                              **metricas_completas(y[mascara], p[mascara])})
resultados = pd.DataFrame(filas)
todas = resultados[resultados.grupo == "todas"]
orden = todas.groupby("modelo")["MAPE (%)"].mean().sort_values().index
METRICAS = [c for c in resultados.columns if c not in ("division", "modelo", "grupo", "ventas", "sin_barrio")]
todas.groupby("modelo")[METRICAS].mean().loc[orden].T.round(2)

modelo,"+ barrio, MAE",Pérdida MAE,+ barrio,"+ barrio, separado",Gradient Boosting actual,Separado,Regresión lineal,Regla ingenua
R²,0.64,0.63,0.61,0.60,0.60,0.60,0.57,0.37
MAE (M),96.77,97.38,102.04,103.27,102.37,103.71,108.92,147.00
RMSE (M),279.81,280.52,278.12,281.67,279.06,282.92,506.94,"1,092.21"
error abs. mediano (M),48.83,48.90,54.91,55.52,54.67,55.96,58.45,65.85
MAPE (%),32.85,32.98,33.49,33.49,33.49,33.53,37.83,56.65


Tabla: promedio de cada métrica en las 20 divisiones, de menor a mayor MAPE.

Gradient Boosting supera con claridad a las dos referencias: su MAPE es de 33,5%, frente a 37,8% de
la regresión lineal y 56,7% de la regla ingenua. Las variantes con pérdida MAE tienen el MAPE más
bajo (32,9% y 33,0%) y el menor error absoluto mediano (49 M frente a 55 M), pero la diferencia es
pequeña y la sección siguiente revisa si es segura.

## ¿Qué tanto depende del azar?

In [5]:
gb_actual = todas[todas.modelo == "Gradient Boosting actual"]
gb_actual[["MAPE (%)", "R²", "error abs. mediano (M)", "ventas"]].describe().loc[["mean", "std", "min", "max"]].round(2)

,MAPE (%),R²,error abs. mediano (M),ventas
mean,33.49,0.60,54.67,"4,317.90"
std,4.19,0.15,11.81,602.04
min,22.85,0.14,34.91,"2,226.00"
max,37.81,0.76,84.24,"4,856.00"


Es el mismo modelo con los mismos datos, y su MAPE va de 22,9% a 37,8% según qué zonas queden
aparte (desviación de 4,2 puntos). Por eso una sola separación no alcanza para concluir nada, y el
resultado de la prueba final puede deberse en buena parte a las zonas que cayeron ahí.

## ¿Qué variante gana?

Para cada variante calculamos, división por división, la diferencia de MAPE con el Gradient
Boosting actual. El intervalo es aproximado porque las divisiones se traslapan, y al comparar varias
variantes alguna puede salir favorecida por azar.

In [6]:
por_division = todas.pivot(index="division", columns="modelo", values="MAPE (%)")
dif = por_division.sub(por_division["Gradient Boosting actual"], axis=0).drop(columns="Gradient Boosting actual")
tabla_dif = pd.DataFrame({"diferencia media": dif.mean(),
                          "IC95 inf": dif.mean() - 2.093 * dif.std() / np.sqrt(len(dif)),
                          "IC95 sup": dif.mean() + 2.093 * dif.std() / np.sqrt(len(dif)),
                          "divisiones en que mejora": (dif < 0).sum()}).sort_values("diferencia media")
tabla_dif.round(2)

,diferencia media,IC95 inf,IC95 sup,divisiones en que mejora
modelo,,,,
"+ barrio, MAE",-0.65,-1.74,0.45,11
Pérdida MAE,-0.52,-1.39,0.35,10
+ barrio,-0.01,-0.45,0.43,9
"+ barrio, separado",-0.00,-0.72,0.72,9
Separado,0.04,-0.70,0.77,11
Regresión lineal,4.33,3.06,5.60,2
Regla ingenua,23.15,19.14,27.17,0


In [7]:
(resultados[resultados.grupo != "todas"]
    .pivot_table(index="modelo", columns="grupo", values=["MAPE (%)", "error abs. mediano (M)"])
    .loc[orden].round(1))

MAPE (%)        error abs. mediano (M)       
grupo                    apartamentos u otros  casas   apartamentos u otros  casas
modelo                                                                            
+ barrio, MAE                           30.80  45.10                  48.10  55.20
Pérdida MAE                             30.90  44.90                  48.30  55.60
+ barrio                                31.80  43.40                  55.30  55.80
+ barrio, separado                      31.50  44.50                  55.50  58.70
Gradient Boosting actual                31.80  43.50                  55.00  56.10
Separado                                31.60  44.00                  56.00  58.20
Regresión lineal                        35.00  53.90                  56.80  70.50
Regla ingenua                           45.00 120.80                  60.20 119.60

In [8]:
print(f"Ventas de validación en un barrio que no aparece en entrenamiento: {todas.sin_barrio.mean():.0f}% en promedio")

Ventas de validación en un barrio que no aparece en entrenamiento: 22% en promedio


Ninguna variante mejora el MAPE de forma segura:

- **Pérdida MAE:** baja el MAPE 0,5 puntos (0,65 si se combina con el barrio), pero su intervalo
  incluye el cero (de −1,4 a +0,4) y mejora solo en 10 de las 20 divisiones.
- **Barrio y modelos separados:** no cambian nada (menos de 0,1 puntos). En promedio, el 22% de
  las ventas de validación está en un barrio que no aparece en entrenamiento.

Las casas siguen cerca de 44-45% de MAPE con todas las variantes, frente a 31% en los
apartamentos. Ajustar el modelo no resuelve eso. Falta información, como el área del lote. Como
ninguna variante mejora de forma segura, nos quedamos con el Gradient Boosting del capítulo 7.

## Validación aleatoria y por zonas

El MAPE de 30,9% del capítulo anterior es menor que el de estas 20 divisiones. Revisamos por qué, y
agregamos una validación aleatoria para ver cuánto cambia el resultado según el diseño.

In [9]:
y_ent = ent.log_precio.to_numpy()

def fuera_de_muestra(particion):
    p = np.empty(len(ent))
    for a, b in particion:
        p[b] = ajustar(ent.iloc[a], ent.iloc[b], BASE)
    return p

p_aleatoria = fuera_de_muestra(KFold(5, shuffle=True, random_state=42).split(ent))
p_bloques = fuera_de_muestra(GroupKFold(5).split(ent, groups=ent.bloque))
bloque_alameda = ent.loc[(ent.barrio == "SECTOR ALAMEDA DEL RIO").fillna(False).to_numpy(dtype=bool), "bloque"].value_counts().index[0]
en_alameda = (ent.bloque == bloque_alameda).to_numpy()
lat, lon = ent.lat.to_numpy(), ent.lon.to_numpy()
pd.DataFrame({
    "Aleatoria": metricas_completas(y_ent, p_aleatoria, lat, lon),
    "Por bloques": metricas_completas(y_ent, p_bloques, lat, lon),
    "Por bloques, sin Alameda": metricas_completas(y_ent[~en_alameda], p_bloques[~en_alameda], lat[~en_alameda], lon[~en_alameda]),
    "Por bloques, solo Alameda": metricas_completas(y_ent[en_alameda], p_bloques[en_alameda]),
}).round(2)

,Aleatoria,Por bloques,"Por bloques, sin Alameda","Por bloques, solo Alameda"
R²,0.80,0.68,0.69,0.35
MAE (M),60.96,89.31,97.15,49.16
RMSE (M),286.23,305.14,332.00,73.65
error abs. mediano (M),22.40,47.01,51.36,26.54
MAPE (%),21.67,30.93,32.34,23.71


- **El diseño pesa mucho.** Con validación aleatoria el MAPE sería de 21,7% y el R² de 0,80; por
  zonas es de 30,9% y 0,68. La validación aleatoria es la que usan muchos estudios, así que sus
  cifras no se comparan directamente con las nuestras. Además tiene una ventaja: la misma manzana o
  el mismo edificio queda a la vez en entrenamiento y en validación. Mide cuánto acierta el modelo
  donde ya hay ventas cercanas, mientras que la validación por zonas mide cuánto acierta donde no
  las hay.
- **El 30,9% es optimista.** El bloque principal de Alameda del Río tiene un MAPE de 23,7%. Sin él,
  la validación por bloques sube a 32,3%.

## Prueba final

Evaluamos una sola vez el modelo elegido, junto con la variante de pérdida MAE y las dos
referencias, en las zonas que dejamos aparte. Con esas zonas calculamos también un intervalo:
remuestreamos los bloques de prueba 2.000 veces.

In [10]:
y_prueba = prueba.log_precio.to_numpy()
la, lo = prueba.lat.to_numpy(), prueba.lon.to_numpy()
predicciones = {"Regla ingenua": ingenua(ent, prueba), "Regresión lineal": lineal(ent, prueba),
                "Gradient Boosting actual": ajustar(ent, prueba, BASE),
                "Gradient Boosting con pérdida MAE": ajustar(ent, prueba, BASE, **MAE)}
pd.DataFrame({k: metricas_completas(y_prueba, p, la, lo) for k, p in predicciones.items()}).round(2)

,Regla ingenua,Regresión lineal,Gradient Boosting actual,Gradient Boosting con pérdida MAE
R²,0.27,0.56,0.62,0.64
MAE (M),98.35,64.63,57.16,52.61
RMSE (M),284.44,240.62,104.83,101.06
error abs. mediano (M),35.25,31.15,34.86,29.54
MAPE (%),58.33,35.56,29.85,29.65


In [11]:
casa = (prueba.tipo_NPH == 1).to_numpy()
pd.concat({g: pd.DataFrame({k: metricas_completas(y_prueba[m], predicciones[k][m])
                            for k in ["Gradient Boosting actual", "Gradient Boosting con pérdida MAE"]})
           for g, m in [("casas", casa), ("apartamentos u otros", ~casa)]}, axis=1).round(2)

casas                                       apartamentos u otros                                  
                       Gradient Boosting actual Gradient Boosting con pérdida MAE Gradient Boosting actual Gradient Boosting con pérdida MAE
R²                                         0.61                              0.61                     0.62                              0.65
MAE (M)                                   90.90                             89.19                    50.29                             45.17
RMSE (M)                                 161.44                            159.66                    89.02                             84.29
error abs. mediano (M)                    52.84                             50.93                    31.59                             25.64
MAPE (%)                                  46.24                             47.97                    26.52                             25.92

In [12]:
pa, pm = predicciones["Gradient Boosting actual"], predicciones["Gradient Boosting con pérdida MAE"]
por_bloque = {b: np.where(prueba.bloque.to_numpy() == b)[0] for b in prueba.bloque.unique()}
nombres_bloque = list(por_bloque)
mape = lambda p, i: np.mean(np.abs(10.0 ** (p[i] - y_prueba[i]) - 1)) * 100
rng = np.random.default_rng(42)
remuestras = []
for _ in range(2000):
    i = np.concatenate([por_bloque[b] for b in rng.choice(nombres_bloque, len(nombres_bloque))])
    remuestras.append((mape(pa, i), mape(pm, i)))
remuestras = np.array(remuestras)
pd.DataFrame({"Gradient Boosting actual": np.percentile(remuestras[:, 0], [2.5, 97.5]),
              "Gradient Boosting con pérdida MAE": np.percentile(remuestras[:, 1], [2.5, 97.5]),
              "Diferencia": np.percentile(remuestras[:, 1] - remuestras[:, 0], [2.5, 97.5])},
             index=["IC95 inf", "IC95 sup"]).round(2)

,Gradient Boosting actual,Gradient Boosting con pérdida MAE,Diferencia
IC95 inf,22.21,22.10,-1.90
IC95 sup,41.41,43.19,2.16


En la prueba final, Gradient Boosting tiene un MAPE de 29,9%, frente a 35,6% de la regresión lineal y 58,3% de la regla ingenua. La pérdida MAE queda prácticamente igual (29,7%): mejora algo a los apartamentos (de 26,5% a 25,9%) y empeora a las casas (de 46,2% a 48,0%). El intervalo de la diferencia va de −1,9 a +2,2 puntos e incluye el cero, lo mismo que en las 20 divisiones. Por eso el modelo final es el Gradient Boosting , sin variantes.